In [1]:
!pip install pandas numpy biopython viennarna

   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 17.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   -------------------- ------------------- 1.0/2.0 MB 16.7 MB/s eta 0:00:01
   -------------------- ------------------- 1.0/2.0 MB 16.7 MB/s eta 0:00:01
   -------------------- ------------------- 1.0/2.0 MB 16.7 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 2.2 MB/s eta 0:00:00


In [2]:
import random
import pandas as pd
import numpy as np
from Bio.SeqUtils import gc_fraction
import RNA

In [3]:
DNA_PARENT = "GCGTGAAGAGGGTAGTCCGACATCGGGAGG"

In [4]:
RNA_PARENT = "AGGUACGACGGUUGGAGGAGUUCGAGUCCG"

In [5]:
DNA_BASES = ['A', 'T', 'G', 'C']
RNA_BASES = ['A', 'U', 'G', 'C']

MUTATION_RATE = 0.12

MIN_GC = 40
MAX_GC = 65

MIN_LENGTH = 30
MAX_LENGTH = 50

MIN_MFE = -5

In [15]:
def get_random_subsequence(parent_seq):

    # Since your parent is already 31 nt,
    # we mainly preserve full sequence length.

    if len(parent_seq) <= MAX_LENGTH:
        return parent_seq

    length = random.randint(MIN_LENGTH, MAX_LENGTH)

    start = random.randint(0, len(parent_seq) - length)

    return parent_seq[start:start+length]

In [16]:
def mutate_sequence(sequence):

    seq_list = list(sequence)

    for i in range(len(seq_list)):

        if random.random() < MUTATION_RATE:

            current_base = seq_list[i]

            possible_bases = [
                b for b in DNA_BASES
                if b != current_base
            ]

            seq_list[i] = random.choice(possible_bases)

    return ''.join(seq_list)

In [17]:
def gc_content_valid(sequence):

    gc = gc_fraction(sequence) * 100

    return MIN_GC <= gc <= MAX_GC

In [18]:
def homopolymer_check(sequence, max_repeat=5):

    count = 1

    for i in range(1, len(sequence)):

        if sequence[i] == sequence[i-1]:

            count += 1

            if count > max_repeat:
                return False

        else:
            count = 1

    return True

In [19]:
def predict_structure(sequence):

    structure, mfe = RNA.fold(sequence)

    return structure, mfe

In [20]:
def structure_valid(structure):

    paired = structure.count('(')

    loops = structure.count('.')

    return paired >= 5 and loops >= 5

In [21]:
def generate_dna_library(parent_sequence,
                         total_sequences):

    library = []

    generated = set()

    while len(library) < total_sequences:

        # Generate subsequence
        subseq = get_random_subsequence(parent_sequence)

        # Introduce mutations
        mutated_seq = mutate_sequence(subseq)

        # Remove duplicates
        if mutated_seq in generated:
            continue

        # GC Content Filter
        if not gc_content_valid(mutated_seq):
            continue

        # Homopolymer Filter
        if not homopolymer_check(mutated_seq):
            continue

        # Secondary Structure Prediction
        structure, mfe = predict_structure(mutated_seq)

        # MFE Filter
        if mfe > MIN_MFE:
            continue

        # Structural Validation
        if not structure_valid(structure):
            continue

        generated.add(mutated_seq)

        library.append({

            'Aptamer_ID':
            f'DNA_{len(library)+1}',

            'Sequence':
            mutated_seq,

            'Length':
            len(mutated_seq),

            'GC_Content':
            round(gc_fraction(mutated_seq) * 100, 2),

            'Structure':
            structure,

            'MFE':
            round(mfe, 2)

        })

    return pd.DataFrame(library)

In [22]:
dna_library = generate_dna_library(
    parent_sequence=DNA_PARENT,
    total_sequences=10000
)


In [23]:
dna_library.to_csv(
    'Ferritin2_DNA_Aptamer_Library_5000.csv',
    index=False
)


In [24]:
print("DNA Aptamer Library Generation Complete!")

print("Total Aptamers Generated:",
      len(dna_library))

print(dna_library.head())

DNA Aptamer Library Generation Complete!
Total Aptamers Generated: 10000
  Aptamer_ID                        Sequence  Length  GC_Content  \
0      DNA_1  GCGTGAAGAGGGTAATCCGACATCGGCAGG      30       60.00   
1      DNA_2  GCGTGATGAGGGTGGTCCGACATCGGGAGA      30       63.33   
2      DNA_3  CCGTGAAGAGGGTAGTCCAACATCGGGACG      30       60.00   
3      DNA_4  GCCTGATGTGGGTAGTCCGACATCGAGAGG      30       60.00   
4      DNA_5  GCGTGAAGATGCAAGTCCGACATCGGGAGG      30       60.00   

                        Structure   MFE  
0  ((.(((.(..((....))..).)))))...  -5.4  
1  .(.(((((..((....))..))))).)...  -9.1  
2  ((.(....).))..((((.......)))).  -5.9  
3  ...((((((.((....)).))))))..... -10.4  
4  ((((....))))...((((....))))...  -7.2  


In [25]:
def random_dna_sequence(length):

    return ''.join(
        random.choice(DNA_BASES)
        for _ in range(length)
    )


In [26]:
def create_extended_sequence(parent_seq):

    parent_length = len(parent_seq)

    # Final desired aptamer length
    final_length = random.randint(
        MIN_LENGTH,
        MAX_LENGTH
    )

    # Extra nucleotides needed
    extra_length = final_length - parent_length

    # Randomly split extensions
    left_extension = random.randint(
        0,
        extra_length
    )

    right_extension = (
        extra_length - left_extension
    )

    # Generate random flanking regions
    left_seq = random_dna_sequence(
        left_extension
    )

    right_seq = random_dna_sequence(
        right_extension
    )

    # Final aptamer
    full_sequence = (
        left_seq +
        parent_seq +
        right_seq
    )

    return full_sequence

In [27]:
def mutate_sequence(sequence):

    seq_list = list(sequence)

    for i in range(len(seq_list)):

        if random.random() < MUTATION_RATE:

            current_base = seq_list[i]

            possible_bases = [
                b for b in DNA_BASES
                if b != current_base
            ]

            seq_list[i] = random.choice(
                possible_bases
            )

    return ''.join(seq_list)

In [28]:
def gc_content_valid(sequence):

    gc = gc_fraction(sequence) * 100

    return MIN_GC <= gc <= MAX_GC

In [29]:
def homopolymer_check(sequence,
                      max_repeat=5):

    count = 1

    for i in range(1, len(sequence)):

        if sequence[i] == sequence[i-1]:

            count += 1

            if count > max_repeat:
                return False

        else:
            count = 1

    return True

In [30]:
def predict_structure(sequence):

    structure, mfe = RNA.fold(sequence)

    return structure, mfe


In [31]:
def structure_valid(structure):

    paired = structure.count('(')

    loops = structure.count('.')

    return paired >= 5 and loops >= 5

In [32]:
def generate_dna_library(parent_sequence,
                         total_sequences):

    library = []

    generated = set()

    while len(library) < total_sequences:

        # Create variable-length sequence
        seq = create_extended_sequence(
            parent_sequence
        )

        # Introduce mutations
        mutated_seq = mutate_sequence(seq)

        # Remove duplicates
        if mutated_seq in generated:
            continue

        # GC Content Filter
        if not gc_content_valid(
            mutated_seq
        ):
            continue

        # Homopolymer Filter
        if not homopolymer_check(
            mutated_seq
        ):
            continue

        # Secondary Structure Prediction
        structure, mfe = predict_structure(
            mutated_seq
        )

        # MFE Filter
        if mfe > MIN_MFE:
            continue

        # Structural Validation
        if not structure_valid(
            structure
        ):
            continue

        generated.add(mutated_seq)

        library.append({

            'Aptamer_ID':
            f'DNA_{len(library)+1}',

            'Sequence':
            mutated_seq,

            'Length':
            len(mutated_seq),

            'GC_Content':
            round(
                gc_fraction(mutated_seq)
                * 100,
                2
            ),

            'Structure':
            structure,

            'MFE':
            round(mfe, 2)

        })

    return pd.DataFrame(library)

In [33]:
dna_library = generate_dna_library(
    parent_sequence=DNA_PARENT,
    total_sequences=10000
)

In [34]:
dna_library.to_csv(
    'Ferritin_DNA_Aptamer_Library_5000.csv',
    index=False
)

In [35]:
print("DNA Aptamer Library Generation Complete!")

print("Total Aptamers Generated:",
      len(dna_library))

print(dna_library.head())

DNA Aptamer Library Generation Complete!
Total Aptamers Generated: 10000
  Aptamer_ID                                           Sequence  Length  \
0      DNA_1  AGACCAGTGGTCTAAGCGTGAATAGGGAAGGCCGACAACAGGAGGGGAT      49   
1      DNA_2     TTAGCGTGAAGAGGGTAGTGCGACATCGGCAGGCACCGATCGCTTC      46   
2      DNA_3  GAAAGTAGCGGCGTTAAGAAGGTATTCCGACATCGGGAGGTTGACC...      50   
3      DNA_4  TGTTCGCAGGTTCTGGGCGTGACGACGGTCGTCCTCCAACGGGAGGAAG      49   
4      DNA_5   TTTCTAGCGTGAAGAGGGTAGTCCGCCATCGGGAGGACGCCGCCTGTG      48   

   GC_Content                                          Structure   MFE  
0       55.10  (((((...)))))...(.((....((.....)).....)).).......  -7.4  
1       58.70     (((.(.(....).).))).((((..((((......))))))))... -10.1  
2       54.00  .......((((.(((((.......(((((....)))))..))))))... -10.0  
3       63.27  ...(((((.(((...))).)).)))......((((((....)))))).. -13.5  
4       62.50   ............((..(((.((((.((....)).)))))))..))... -13.7  


In [36]:
RNA_PARENT = "AGGUACGACGGUUGGAGGAGUUCGAGUCCG"


In [37]:
RNA_BASES = ['A', 'U', 'G', 'C']

MUTATION_RATE = 0.12

MIN_GC = 40
MAX_GC = 65

MIN_LENGTH = 30
MAX_LENGTH = 50

MIN_MFE = -5

In [38]:
def random_rna_sequence(length):

    return ''.join(
        random.choice(RNA_BASES)
        for _ in range(length)
    )

In [39]:
def create_extended_sequence(parent_seq):

    parent_length = len(parent_seq)

    # Final desired aptamer length
    final_length = random.randint(
        MIN_LENGTH,
        MAX_LENGTH
    )

    # Extra nucleotides needed
    extra_length = final_length - parent_length

    # Randomly split extensions
    left_extension = random.randint(
        0,
        extra_length
    )

    right_extension = (
        extra_length - left_extension
    )

    # Generate random flanking regions
    left_seq = random_rna_sequence(
        left_extension
    )

    right_seq = random_rna_sequence(
        right_extension
    )

    # Final aptamer
    full_sequence = (
        left_seq +
        parent_seq +
        right_seq
    )

    return full_sequence

In [40]:
def mutate_sequence(sequence):

    seq_list = list(sequence)

    for i in range(len(seq_list)):

        if random.random() < MUTATION_RATE:

            current_base = seq_list[i]

            possible_bases = [
                b for b in RNA_BASES
                if b != current_base
            ]

            seq_list[i] = random.choice(
                possible_bases
            )

    return ''.join(seq_list)

In [41]:
def gc_content_valid(sequence):

    gc = gc_fraction(sequence) * 100

    return MIN_GC <= gc <= MAX_GC

In [42]:
def homopolymer_check(sequence,
                      max_repeat=5):

    count = 1

    for i in range(1, len(sequence)):

        if sequence[i] == sequence[i-1]:

            count += 1

            if count > max_repeat:
                return False

        else:
            count = 1

    return True

In [43]:
def predict_structure(sequence):

    structure, mfe = RNA.fold(sequence)

    return structure, mfe


In [44]:
def structure_valid(structure):

    paired = structure.count('(')

    loops = structure.count('.')

    return paired >= 5 and loops >= 5

In [45]:
def generate_rna_library(parent_sequence,
                         total_sequences):

    library = []

    generated = set()

    while len(library) < total_sequences:

        # Create variable-length sequence
        seq = create_extended_sequence(
            parent_sequence
        )

        # Introduce mutations
        mutated_seq = mutate_sequence(seq)

        # Remove duplicates
        if mutated_seq in generated:
            continue

        # GC Content Filter
        if not gc_content_valid(
            mutated_seq
        ):
            continue

        # Homopolymer Filter
        if not homopolymer_check(
            mutated_seq
        ):
            continue

        # Secondary Structure Prediction
        structure, mfe = predict_structure(
            mutated_seq
        )

        # MFE Filter
        if mfe > MIN_MFE:
            continue

        # Structural Validation
        if not structure_valid(
            structure
        ):
            continue

        generated.add(mutated_seq)

        library.append({

            'Aptamer_ID':
            f'RNA_{len(library)+1}',

            'Sequence':
            mutated_seq,

            'Length':
            len(mutated_seq),

            'GC_Content':
            round(
                gc_fraction(mutated_seq)
                * 100,
                2
            ),

            'Structure':
            structure,

            'MFE':
            round(mfe, 2)

        })

    return pd.DataFrame(library)

In [46]:
rna_library = generate_rna_library(
    parent_sequence=RNA_PARENT,
    total_sequences=10000
)

In [47]:
rna_library.to_csv(
    'Ferritin_RNA_Aptamer_Library_5000.csv',
    index=False
)

In [48]:
print("RNA Aptamer Library Generation Complete!")

print("Total Aptamers Generated:",
      len(rna_library))

print(rna_library.head())

RNA Aptamer Library Generation Complete!
Total Aptamers Generated: 10000
  Aptamer_ID                                           Sequence  Length  \
0      RNA_1           GUUUUCGUAAGGUACGACGGGUGGAGGAGUUCGGGUCCGG      40   
1      RNA_2  GCAUAAGCAGAGGUACGACGGUUGGAGGCGUUCGGGUCCGCCCGUAUCA      49   
2      RNA_3      UUUAGGCGGAGGUACGACGGUUGGAUGUGUUACAGUCCGCGGCCC      45   
3      RNA_4                     AGGUGCGACGGUUCGAGGAGUUCGACUCCG      30   
4      RNA_5    GAAACCUGACGUACGACGGUUGGAUGAGUUCGAGUCCGAAAGCGUUC      47   

   GC_Content                                          Structure   MFE  
0       60.00           ....(((((...)))))((((((((....))))..)))).  -7.3  
1       61.22  ((....))...((((((.(((((.((.....)).)).)))..)))))). -12.5  
2       60.00      .....(((((.(((..(((......)))..)))..)))))..... -12.9  
3       63.33                     .....(((....))).(((((...))))).  -6.2  
4       53.19    .......(((((....(((((.((.....)).)).)))...))))).  -9.3  
